In [1]:
import pandas as pd

print("TalentMatch - Data Understanding")
print("Pandas version:", pd.__version__)

TalentMatch - Data Understanding
Pandas version: 3.0.6


In [3]:
import pandas as pd
from pathlib import Path

# Project folder
project_path = Path("/Users/omnia/Capstone_TalentMatch")

# Jobs dataset path
jobs_path = project_path / "data" / "jobs" / "jobs.tsv"

# Read only 5 rows
jobs = pd.read_csv(
    jobs_path,
    sep="\t",
    nrows=5
)

display(jobs)

,JobID,WindowID,Title,Description,Requirements,City,State,Country,Zip5,StartDate,EndDate
0,1,1,Security Engineer/Technical Lead,<p>Security Clearance Required:&nbsp; Top Secr...,<p>SKILL SET</p>\r<p>&nbsp;</p>\r<p>Network Se...,Washington,DC,US,20531.0,2012-03-07 13:17:01.643,2012-04-06 23:59:59
1,4,1,SAP Business Analyst / WM,<strong>NO Corp. to Corp resumes&nbsp;are bein...,<p><b>WHAT YOU NEED: </b></p>\r<p>Four year co...,Charlotte,NC,US,28217.0,2012-03-21 02:03:44.137,2012-04-20 23:59:59
2,7,1,P/T HUMAN RESOURCES ASSISTANT,<b> <b> P/T HUMAN RESOURCES ASSISTANT</b> <...,Please refer to the Job Description to view th...,Winter Park,FL,US,32792.0,2012-03-02 16:36:55.447,2012-04-01 23:59:59
3,8,1,Route Delivery Drivers,CITY BEVERAGES Come to work for the best in th...,Please refer to the Job Description to view th...,Orlando,FL,US,NaN,2012-03-03 09:01:10.077,2012-04-02 23:59:59
4,9,1,Housekeeping,I make sure every part of their day is magica...,Please refer to the Job Description to view th...,Orlando,FL,US,NaN,2012-03-03 09:01:11.88,2012-04-02 23:59:59


In [4]:
# Read the first 1000 jobs
jobs_sample = pd.read_csv(
    jobs_path,
    sep="\t",
    nrows=1000
)

# Dataset size
print("Dataset Shape:", jobs_sample.shape)

# Missing values
print("\nMissing Values:")
display(jobs_sample.isnull().sum())

Dataset Shape: (1000, 11)

Missing Values:


JobID             0
WindowID          0
Title             0
Description       0
Requirements      3
City              0
State             0
Country           0
Zip5            312
StartDate         0
EndDate           0
dtype: int64

In [5]:
# Check duplicate jobs
print("Total Jobs:", len(jobs_sample))

# Exact duplicate rows
print("Duplicate Rows:", jobs_sample.duplicated().sum())

# Duplicate JobIDs within the same window
print(
    "Duplicate Job IDs:",
    jobs_sample.duplicated(
        subset=["WindowID", "JobID"]
    ).sum()
)

Total Jobs: 1000
Duplicate Rows: 0
Duplicate Job IDs: 0


In [6]:
import re
import html

# Function to clean HTML text
def clean_text(text):
    if pd.isna(text):
        return ""

    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# Create a copy of the sample
jobs_clean = jobs_sample.copy()

# Clean text columns
for col in ["Title", "Description", "Requirements"]:
    jobs_clean[col] = jobs_clean[col].apply(clean_text)

# Compare before and after cleaning
print("BEFORE CLEANING:")
print(jobs_sample["Description"].iloc[0][:300])

print("\nAFTER CLEANING:")
print(jobs_clean["Description"].iloc[0][:300])

BEFORE CLEANING:
<p>Security Clearance Required:&nbsp; Top Secret </p>\r<p>Job Number: TMR-447</p>\r<p>Location of Job:&nbsp; Washington, DC</p>\r<p>TMR, Inc. is an Equal Employment Opportunity Company</p>\r<p>For more job opportunities with TMR, visit our website <a href="http://www.tmrhq.com/">www.tmrhq.com</a></p

AFTER CLEANING:
Security Clearance Required: Top Secret \r Job Number: TMR-447 \r Location of Job: Washington, DC \r TMR, Inc. is an Equal Employment Opportunity Company \r For more job opportunities with TMR, visit our website www.tmrhq.com \r Send Resumes to HR@tmrhq2.com \r \r JOB SUMMARY: \r \r Leads the custom


In [7]:
# Remove remaining escaped line breaks and extra spaces
for col in ["Title", "Description", "Requirements"]:
    jobs_clean[col] = (
        jobs_clean[col]
        .str.replace(r"\\[rnt]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Check the final cleaned description
print("FINAL CLEANED TEXT:")
print(jobs_clean["Description"].iloc[0][:500])

FINAL CLEANED TEXT:
Security Clearance Required: Top Secret Job Number: TMR-447 Location of Job: Washington, DC TMR, Inc. is an Equal Employment Opportunity Company For more job opportunities with TMR, visit our website www.tmrhq.com Send Resumes to HR@tmrhq2.com JOB SUMMARY: Leads the customer’s overall Cyber Security strategy, formalizes service offerings consisted with ITIL best practices, and provides design and architecture support. Provide security design / architecture support for OJP’s IT Security Division 


In [8]:
# Combine job information into one text column
jobs_clean["job_text"] = (
    jobs_clean["Title"].fillna("") + " " +
    jobs_clean["Description"].fillna("") + " " +
    jobs_clean["Requirements"].fillna("")
).str.strip()

# Check the result
print("Total Jobs:", len(jobs_clean))

display(
    jobs_clean[["JobID", "Title", "job_text"]]
    .head(5)
    .assign(job_text=lambda x: x["job_text"].str[:150])
)

Total Jobs: 1000


,JobID,Title,job_text
0,1,Security Engineer/Technical Lead,Security Engineer/Technical Lead Security Clea...
1,4,SAP Business Analyst / WM,SAP Business Analyst / WM NO Corp. to Corp res...
2,7,P/T HUMAN RESOURCES ASSISTANT,P/T HUMAN RESOURCES ASSISTANT P/T HUMAN RESOUR...
3,8,Route Delivery Drivers,Route Delivery Drivers CITY BEVERAGES Come to ...
4,9,Housekeeping,Housekeeping I make sure every part of their d...


In [9]:
# Load a sample of users
users_path = project_path / "data" / "users.tsv"

users_sample = pd.read_csv(
    users_path,
    sep="\t",
    nrows=1000
)

# Check dataset shape
print("Users Dataset Shape:", users_sample.shape)

# Display first 5 users
display(users_sample.head(5))

Users Dataset Shape: (1000, 15)


,UserID,WindowID,Split,City,State,Country,ZipCode,DegreeType,Major,GraduationDate,WorkHistoryCount,TotalYearsExperience,CurrentlyEmployed,ManagedOthers,ManagedHowMany
0,47,1,Train,Paramount,CA,US,90723,High School,NaN,1999-06-01 00:00:00,3,10.0,Yes,No,0
1,72,1,Train,La Mesa,CA,US,91941,Master's,Anthropology,2011-01-01 00:00:00,10,8.0,Yes,No,0
2,80,1,Train,Williamstown,NJ,US,08094,High School,Not Applicable,1985-06-01 00:00:00,5,11.0,Yes,Yes,5
3,98,1,Train,Astoria,NY,US,11105,Master's,Journalism,2007-05-01 00:00:00,3,3.0,Yes,No,0
4,123,1,Train,Baton Rouge,LA,US,70808,Bachelor's,Agricultural Business,2011-05-01 00:00:00,1,9.0,Yes,No,0


In [10]:
# Check missing values in users data
print("Missing Values:")
display(users_sample.isnull().sum())

# Check duplicate users
print("Duplicate Rows:", users_sample.duplicated().sum())

# Check dataset splits
print("\nData Splits:")
display(users_sample["Split"].value_counts())

Missing Values:


UserID                    0
WindowID                  0
Split                     0
City                      0
State                     2
Country                   0
ZipCode                  14
DegreeType              248
Major                   240
GraduationDate          293
WorkHistoryCount          0
TotalYearsExperience     33
CurrentlyEmployed       141
ManagedOthers             0
ManagedHowMany            0
dtype: int64

Duplicate Rows: 0

Data Splits:


Split
Train    939
Test      61
Name: count, dtype: int64

In [11]:
# Check duplicate users
print("Duplicate Rows:", users_sample.duplicated().sum())

# Check data splits
print("\nData Splits:")
display(users_sample["Split"].value_counts())

Duplicate Rows: 0

Data Splits:


Split
Train    939
Test      61
Name: count, dtype: int64

In [12]:
# Load applications dataset
apps_path = project_path / "data" / "apps" / "apps.tsv"

apps_sample = pd.read_csv(
    apps_path,
    sep="\t",
    nrows=1000
)

# Dataset shape
print("Applications Shape:", apps_sample.shape)

# Display first 5 applications
display(apps_sample.head())

# Check Train/Test distribution
print("\nData Splits:")
display(apps_sample["Split"].value_counts())

Applications Shape: (1000, 5)


,UserID,WindowID,Split,ApplicationDate,JobID
0,47,1,Train,2012-04-04 15:56:23.537,169528
1,47,1,Train,2012-04-06 01:03:00.003,284009
2,47,1,Train,2012-04-05 02:40:27.753,2121
3,47,1,Train,2012-04-05 02:37:02.673,848187
4,47,1,Train,2012-04-05 22:44:06.653,733748



Data Splits:


Split
Train    850
Test     150
Name: count, dtype: int64

In [13]:
print("Total Applications:", len(apps_sample))
print("Unique Users:", apps_sample["UserID"].nunique())
print("Unique Jobs:", apps_sample["JobID"].nunique())
print("Duplicate Applications:", apps_sample.duplicated().sum())

print("\nApplications per Window:")
display(apps_sample["WindowID"].value_counts())

Total Applications: 1000
Unique Users: 215
Unique Jobs: 978
Duplicate Applications: 0

Applications per Window:


WindowID
1    1000
Name: count, dtype: int64

In [14]:
# Step 14: ESCO Dataset Exploration

esco_path = (
    project_path
    / "data"
    / "ESCO dataset - v1.2.1 - classification - en - csv"
)

occupations = pd.read_csv(
    esco_path / "occupations_en.csv",
    nrows=5
)

skills = pd.read_csv(
    esco_path / "skills_en.csv",
    nrows=5
)

relations = pd.read_csv(
    esco_path / "occupationSkillRelations_en.csv",
    nrows=5
)

print("Occupations Columns:")
print(occupations.columns.tolist())

print("\nSkills Columns:")
print(skills.columns.tolist())

print("\nRelations Columns:")
print(relations.columns.tolist())

Occupations Columns:
['conceptType', 'conceptUri', 'iscoGroup', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'regulatedProfessionNote', 'scopeNote', 'definition', 'inScheme', 'description', 'code', 'naceCode']

Skills Columns:
['conceptType', 'conceptUri', 'skillType', 'reuseLevel', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'scopeNote', 'definition', 'inScheme', 'description']

Relations Columns:
['occupationUri', 'occupationLabel', 'relationType', 'skillType', 'skillUri', 'skillLabel']


In [15]:
# Step 15: Explore Occupation-Skill Relationships

display(
    relations[
        ["occupationLabel", "relationType", "skillLabel"]
    ].head(5)
)

,occupationLabel,relationType,skillLabel
0,technical director,essential,theatre techniques
1,technical director,essential,organise rehearsals
2,technical director,essential,write risk assessment on performing arts produ...
3,technical director,essential,coordinate with creative departments
4,technical director,essential,adapt to artists' creative demands


In [16]:
# Load the full ESCO relations dataset
relations_full = pd.read_csv(
    esco_path / "occupationSkillRelations_en.csv"
)

# Search for Data Analyst and Data Scientist
data_jobs = relations_full[
    relations_full["occupationLabel"].str.contains(
        "data analyst|data scientist",
        case=False,
        na=False
    )
]

print("Matching Relations:", len(data_jobs))

# Show related skills
display(
    data_jobs[
        ["occupationLabel", "relationType", "skillLabel"]
    ].head(15)
)

Matching Relations: 164


,occupationLabel,relationType,skillLabel
19748,data scientist,essential,mathematical modelling
19749,data scientist,essential,data mining
19750,data scientist,essential,visual presentation techniques
19751,data scientist,essential,data engineering
19752,data scientist,essential,quantitative analysis
19753,data scientist,essential,data visualisation software
19754,data scientist,essential,statistical modeling techniques
19755,data scientist,essential,information extraction
19756,data scientist,essential,scientific literature
19757,data scientist,essential,statistics


In [17]:
# Step 17: Match User Job History with ESCO

# Load a sample of user job history
history_sample = pd.read_csv(
    project_path / "data" / "user_history.tsv",
    sep="\t",
    nrows=1000
)

# Get ESCO occupation names
esco_titles = set(
    relations_full["occupationLabel"]
    .dropna()
    .str.strip()
    .str.casefold()
)

# Check exact occupation matches
history_sample["esco_match"] = (
    history_sample["JobTitle"]
    .fillna("")
    .str.strip()
    .str.casefold()
    .isin(esco_titles)
)

print("Total History Records:", len(history_sample))
print("Matched with ESCO:", history_sample["esco_match"].sum())

display(
    history_sample[["JobTitle", "esco_match"]].head(10)
)

Total History Records: 1000
Matched with ESCO: 99


,JobTitle,esco_match
0,National Space Communication Programs-Special ...,False
1,Detention Officer,False
2,"Passenger Screener, TSA",False
3,"Lecturer, Department of Anthropology",False
4,Student Assistant,False
5,Elderly Caregiver,False
6,NaN,False
7,"Department Assistant, Department",False
8,Book Reviews Editorial Assistant,False
9,Graduate Assistant,False


In [18]:
# Step 18: ESCO Exact Matching Results

total = len(history_sample)
matched = history_sample["esco_match"].sum()

print("Total Records:", total)
print("Matched Records:", matched)
print("Unmatched Records:", total - matched)
print("Match Rate:", round(matched / total * 100, 2), "%")

Total Records: 1000
Matched Records: 99
Unmatched Records: 901
Match Rate: 9.9 %
